In [ ]:
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import numpy as np


from matplotlib import pyplot as plt
from matplotlib import colormaps, colors, transforms
from matplotlib.patches import Rectangle, FancyArrow, Arc
from matplotlib.lines import Line2D
import matplotlib as mpl
import seaborn as sns
import colorcet
import colorsys

from Bio import SeqIO, Seq, SeqRecord, Restriction, Phylo
from phytreeviz import TreeViz
# Dunno why phytreeviz changes these at import, took me forever to find it. Revert back to values.
mpl_rc_params = {
    "svg.fonttype": "path",
    "savefig.bbox": "standard",
}
mpl.rcParams.update(mpl_rc_params)

import re



from scipy.cluster import hierarchy
from scipy import stats
from scipy.spatial.distance import euclidean, pdist, squareform
from sklearn.metrics import pairwise_distances, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import AgglomerativeClustering, KMeans

import itertools

from progressbar import ProgressBar

from genomicranges import GenomicRanges
from iranges import IRanges
from biocframe import BiocFrame

import sys
sys.path.append('./')
import variables as v

import mathieu as mh
import importlib

fig_path = f'/home/mathieu/postdoc_heasley/publications/subtelomere_paper/fig/'

In [ ]:
importlib.reload(v)

In [ ]:
wine_subclade = [f'YJM{i}' for i in (947, 948, 954, 955, 956, 957, 963, 964, 965, 967)]

# Annotations of Fisher long-read assemblies

In [ ]:
GFF = {}
for s in v.ena_accessions:
    # Y prime elements
    try:
        gff_yprime = v.parse_gff(f'/home/mathieu/postdoc_heasley/long_read_project/assemblies_fisher/{s}/{s}.nuclear_genome.Y_prime_element.gff3')
    except pd.errors.EmptyDataError:
        print(f'{s} has no Y prime')
        gff_yprime = pd.DataFrame([])

    # TG repeat
    try:
        gff_tg = v.parse_gff(f'/home/mathieu/postdoc_heasley/long_read_project/assemblies_fisher/{s}/{s}.nuclear_genome.TG_repeat.gff3')
    except pd.errors.EmptyDataError:
        print(f'{s} has no TG repeats')
        gff_tg = pd.DataFrame([])
    
    GFF[s] = pd.concat([gff_yprime, gff_tg], axis=0).reset_index(drop=True)

# Import MSA as str, int and bool

In [ ]:
# Import MAFFT trimmed alignment

aln_trim_seq = {}

# trimmed alignment

for seq in SeqIO.parse(f'{v.base_dir}/yprime_msa/combined.yprime.underscore.aln.trim.fasta', 'fasta'):
    aln_trim_seq[seq.id] = np.array(list(seq.seq))

aln_trim_seq = pd.DataFrame(aln_trim_seq).T
n_seqs = aln_trim_seq.shape[0]

aln_trim_bool = aln_trim_seq!='-'

# Compute distance matrices

In [ ]:
#distance_matrix_seq = pdist(aln_trim_seq.values, metric=lambda u,v: (u!=v).sum())
#np.save(f'{base_dir}tables/yprime_pairwise_distance.npy', distance_matrix_seq)
distance_matrix_seq = np.load(f'{v.base_dir}tables/yprime_pairwise_distance.npy')
square_distance_matrix_seq = squareform(distance_matrix_seq)

In [ ]:
linkages = {}
for m in ['weighted', 'complete', 'average']:
    L = hierarchy.linkage(distance_matrix_seq, method=m, metric=None)
    linkages[m] = L

# Y' element MSA clustering optimization

## Agglomerative clustering
### Evaluate distance metrics and distance thresholds
#### Sequence-based distances

In [ ]:
for m, d in itertools.product(linkages.keys(), np.arange(0.1, 0.8, 0.05)):

    L = linkages[m]
    t = d*L[:,2].max()

    fig, axes = plt.subplots(ncols=2, figsize=[12,12], gridspec_kw=dict(width_ratios=[1,3]))

    ax = axes[0]
    
    D = hierarchy.dendrogram(L, orientation='left', color_threshold=t, ax=ax)
    n_clusters = (np.unique(hierarchy.fcluster(L, t=t, criterion='distance'), return_counts=True)[1]>1).sum()

    for coll in ax.collections:
        coll.set_linewidth(0.2)
    ax.axvline(t, c='red', ls='--', lw=0.5)
    ax.set_yticks([])

    ax = axes[1]
    ax.imshow(aln_trim_num.iloc[D['leaves']], cmap=atcg_cmap, aspect='auto', interpolation='nearest')
    ax.invert_yaxis()
    ax.set_title(fr'{m} t={d:.2f}$\times$max {n_clusters:.0f} clusters')

    plt.savefig(f'{fig_path}subtelomeres/yprime_cluster_opt_{m}_{d:.2f}.png', dpi=500)
    #plt.show()
    plt.close()

In [ ]:
for dt in np.logspace(np.log10(10), np.log10(10000), 8):
    
    agcl = AgglomerativeClustering(n_clusters=None, metric='precomputed', linkage=l, distance_threshold=dt)
    agcl_fit_predict = agcl.fit_predict(square_distance_matrix_seq)

    n_clusters = np.unique(agcl_fit_predict).shape[0]
    
    if 2 <= n_clusters <= n_seqs-1:
        
        sils = silhouette_score(square_distance_matrix_seq, agcl_fit_predict, metric='precomputed')

    else:
        sils = np.nan

    seq_agg_cluster_evaluation.append([l, dt, n_clusters, sils])

seq_agg_cluster_evaluation = pd.DataFrame(seq_agg_cluster_evaluation, columns=['linkage', 'dist', 'n_clusters', 'silhouette'])

In [ ]:
fig, ax = plt.subplots(figsize=[7,7])

for l, df in seq_agg_cluster_evaluation.groupby('linkage'):
    
    ax.plot(df['dist'], df['n_clusters'], label=l)
    ax.scatter(df['dist'], df['n_clusters'], c=df['silhouette'].apply(lambda x: colormaps['viridis']((x+1)/2)))
    
ax.set_xscale('log')
ax.legend()
ax.set_yscale('log')

plt.show()
plt.close()

In [ ]:
seq_agg_cluster_evaluation.loc[(seq_agg_cluster_evaluation['n_clusters']<20) &
                                (seq_agg_cluster_evaluation['n_clusters']>1)].sort_values(by='n_clusters')

#### Indel-based distances

In [ ]:
distance_metrics = ['cityblock', 'euclidean', 'cosine']
distance_matrices = {}
for m in distance_metrics:
    dm = pairwise_distances(aln_int, metric=m)
    distance_matrices[m] = dm

In [ ]:
metrics_dist_range = {m: np.logspace(np.log10(np.quantile(mx.flatten(), 0.05)), np.log10(np.quantile(mx.flatten(), 0.95)), 50) for m, mx in distance_matrices.items()}

In [ ]:
agg_cluster_evaluation = []

idx = 0
with ProgressBar(max_value=150) as bar:

    for m in distance_metrics:
        dist_matrix = distance_matrices[m]
        distance_thresholds = metrics_dist_range[m]
        
        for dt in distance_thresholds:
            agcl = AgglomerativeClustering(n_clusters=None, metric='precomputed', linkage='average', distance_threshold=dt)
            agcl_fit_predict = agcl.fit_predict(dist_matrix)
    
            n_clusters = np.unique(agcl_fit_predict).shape[0]
    
            if 2 <= n_clusters <= n_seqs-1:
                
                sils = silhouette_score(dist_matrix, agcl_fit_predict, metric='precomputed')
                dbs = davies_bouldin_score(aln_int, agcl_fit_predict)
                chs = calinski_harabasz_score(aln_int, agcl_fit_predict)

            else:
                sils = np.nan
                dbs = np.nan
                chs = np.nan
    
            agg_cluster_evaluation.append([m, dt, n_clusters, sils, dbs])

            idx += 1
            bar.update(idx)

agg_cluster_evaluation = pd.DataFrame(agg_cluster_evaluation, columns=['dist_metric', 'dist_threshold', 'n_clusters', 'silhouette', 'davies-bouldin'])

In [ ]:
#cluster_evaluation.to_csv(f'{base_dir}tables/y_prime_cluster_evaluation.csv', index=False)

In [ ]:
metric_axes = dict(zip(distance_metrics, range(3)))
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=[10,10])

for m, df in agg_cluster_evaluation.groupby('dist_metric'):
    ax = axes[metric_axes[m], 0]
    sns.scatterplot(x='dist_threshold', y='silhouette', hue='n_clusters', data=df, ax=ax)
    ax.set_title(m)
    ax.set_ylabel('Silhouette score')
    
    ax = axes[metric_axes[m], 1]
    sns.scatterplot(x='dist_threshold', y='davies-bouldin', hue='n_clusters', data=df, ax=ax)
    ax.set_title(m)
    ax.set_ylabel('Davies-Bouldin score')

plt.tight_layout()
plt.show()
plt.close()

### Perform agglomerative clustering with optimal parameters

In [ ]:
agcl = AgglomerativeClustering(n_clusters=None, metric='precomputed', linkage='average', distance_threshold=1600)
agcl_fit_predict = agcl.fit_predict(distance_matrices['cityblock'])

## KMeans
### Evaluate number of clusters

In [ ]:
kmean_evaluation = []

idx = 0
with ProgressBar(max_value=199) as bar:
    for n in range(2,100):
    
        km = KMeans(n_clusters=n, random_state=42)
        km_fit_predict = km.fit_predict(aln_int)
        
        sils = silhouette_score(distance_matrices['euclidean'], km_fit_predict, metric='precomputed')
        dbs = davies_bouldin_score(aln_int, km_fit_predict)
        chs = calinski_harabasz_score(aln_int, km_fit_predict)
    
        kmean_evaluation.append([n, sils, dbs, chs])

        idx += 1
        bar.update(idx)

kmean_evaluation = pd.DataFrame(kmean_evaluation, columns=['n_clusters', 'silhouette', 'davies-bouldin', 'calinski_harabasz'])

In [ ]:
fig, axes = plt.subplots(nrows=3)
ax = axes[0]
sns.scatterplot(x='n_clusters', y='silhouette', data=kmean_evaluation, color='k', ax=ax)

ax = axes[1]
sns.scatterplot(x='n_clusters', y='davies-bouldin', data=kmean_evaluation, color='r', ax=ax)

ax = axes[2]
sns.scatterplot(x='n_clusters', y='calinski_harabasz', data=kmean_evaluation, color='b', ax=ax)

for ax in axes:
    ax.set_xlim(0, 20)
plt.show()
plt.close()

# Clustering with optimal parameters

In [ ]:
m = 'average'
d = 0.1
L = linkages['average']
dt = 0.1*L[:,2].max()
hclust = hierarchy.fcluster(L, t=dt, criterion='distance')

In [ ]:
# from the optimal clustering, build table of sequence to type
yprime_seq_order = pd.DataFrame([hclust, aln_trim_seq.index], index=['cluster','name']).T
# add yprime types for major clusters, at least 100 sequences
type_idx = 1
for yt, df in yprime_seq_order.groupby('cluster'):
    if df.shape[0] >= 100:
        yprime_seq_order.loc[df.index, 'type'] = type_idx
        type_idx += 1
# exclude non-type sequences
yprime_seq_order = yprime_seq_order.loc[~yprime_seq_order['type'].isna()].astype({'type':np.int32})

# from query names, extract strain and alignment info
pattern_qry_aln = re.compile(r'(.*)_(\d+)-(\d+)\(([+-])\)')
for name, df in yprime_seq_order.groupby('name'):
    strain_assembly, aln_yprime, aln_qry = name.split('__')
    if strain_assembly == 'YJM1448_SFE':
        strain_assembly = 'YJM1448'
    strain = v.ena_to_std_dict.get(strain_assembly, strain_assembly)
    query, start, end, strand = re.match(pattern_qry_aln, aln_qry).groups()
    
    yprime_seq_order.loc[df.index, 'strain_assembly'] = strain_assembly
    yprime_seq_order.loc[df.index, 'strain'] = strain
    yprime_seq_order.loc[df.index, 'query'] = query
    yprime_seq_order.loc[df.index, 'start'] = start
    yprime_seq_order.loc[df.index, 'end'] = end
    yprime_seq_order.loc[df.index, 'strand'] = strand
    
yprime_seq_order = yprime_seq_order.astype({'start':np.int32, 'end':np.int32})
yprime_seq_order['len'] = yprime_seq_order['end']-yprime_seq_order['start']
# exclude JAY270
yprime_seq_order = yprime_seq_order.loc[yprime_seq_order['strain']!='JAY270']
# write yprime type full names
yprime_seq_order['yprime_type'] = yprime_seq_order['type'].apply(lambda x: f'yprime.{x}.trim_cons')
# final sorting
yprime_seq_order = yprime_seq_order.sort_values(by=['type', 'strain', 'name'])

In [ ]:
yprime_types_N = yprime_seq_order['type'].unique().shape[0]
yprime_types_int = [int(yt.split('.')[1]) for yt in v.yprime_types] + [14,15,16,17]
yprime_types_rank = dict(zip(yprime_types_int, [yprime_types_int.index(i)+1 for i in sorted(yprime_types_int)]))
yprime_types_colors = [colorcet.glasbey_warm[yprime_types_rank[i]] for i in yprime_types_int]
yprime_types_cmap = colors.LinearSegmentedColormap.from_list('yprime_types', yprime_types_colors, N=yprime_types_N)

clades_unique_N = v.clades_unique.shape[0]
clades_unique_idx = np.arange(clades_unique_N)
clades_colors = v.clades_unique['mfc'].values
clades_cmap = colors.LinearSegmentedColormap.from_list('clades', clades_colors, N=clades_unique_N)

## Build collection combined metadata table

In [ ]:
# In-house YJM collection
strains_info_sr = pd.read_csv('/home/mathieu/postdoc_heasley/short_read_project/script/strains_info.csv').set_index('strain', drop=False)

#collection_yprime = yprime_seq_order.value_counts(['strain_assembly', 'strain']).rename('y_prime_cn_estimate').reset_index()

collection_yprime = pd.concat([yprime_seq_order.value_counts(['strain_assembly', 'strain']).rename('y_prime_cn_estimate'), 
                               yprime_seq_order.groupby(['strain_assembly', 'strain'])['len'].sum().rename('y_prime_len')], axis=1)\
    .reset_index().merge(strains_info_sr[['source_simplified', 'ploidy', 'clade']],
                         left_on='strain', right_index=True, how='left').set_index('strain_assembly', drop=False)

#collection_yprime = collection_yprime.merge(strains_info_sr[['source_simplified', 'ploidy', 'clade']],
#                                           left_on='strain', right_index=True, how='left').set_index('strain_assembly', drop=False)

# Fill clade info for 1011 strains
for i, (sa, s, cn, yl, source, ploidy, clade) in collection_yprime.loc[collection_yprime['clade'].isna()].iterrows():
    if s in v.strains_1011.index:
        clade = v.strains_1011.loc[s, 'Clades']
    elif s == 'AIS':
        clade = '1. Wine/European'
    else:
        clade = 'Unclustered'
    collection_yprime.loc[i, 'clade'] = clade
    
collection_yprime['clade_no'] = v.clades_unique.loc[collection_yprime['clade'], 'clade_no'].values
collection_yprime['raw_reads'] = collection_yprime['strain'].apply(lambda s: s in v.strains_info.index)
# correct CN for raw reads, which are 5X coverage
for c in ['y_prime_cn_estimate', 'y_prime_len']:
    collection_yprime[c] = np.where(collection_yprime['raw_reads'],
                                    collection_yprime[c]/5,
                                    collection_yprime[c])

strain_clade_dict = collection_yprime.groupby('strain')['clade'].apply(lambda x: x.unique()[0]).to_dict()
strain_source_dict = collection_yprime.groupby('strain')['source_simplified'].apply(lambda x: x.unique()[0]).to_dict()

## Fig S3A

In [ ]:
S_wine_subclade = [f'YJM{i}' for i in (947,948,954,955,956,957,963,964,965,967)] + ['ADI', 'CASBLI']

In [ ]:
fig = plt.figure(figsize=[6,12])
gs = plt.GridSpec(ncols=5, nrows=1, width_ratios=[12,1,1,1,1], wspace=0.1,
                 left=0.12, right=0.96, top=0.92, bottom=0.04)

S_seq_order = yprime_seq_order.loc[~yprime_seq_order['type'].isna(), 'strain_assembly'].values
I = yprime_seq_order['name'].values

ax = fig.add_subplot(gs[0])
dat = aln_trim_seq.loc[I].map(lambda x: v.atcg_num[x])
ax.imshow(dat, cmap=v.atcg_cmap, aspect='auto')

Y = np.arange(0, len(I), 500)+1
ax.set_yticks(Y)
ax.set_yticklabels(Y, size=7)
ax.set_ylabel('# sequences')
ax.set_xlabel('coord (bp)')
for sp in ax.spines.values():
    sp.set_visible(False)

# Y prime type (new)
ax = fig.add_subplot(gs[1])
dat = yprime_seq_order['type'].astype(int).values.reshape(-1,1)
ax.imshow(dat, cmap=yprime_types_cmap, aspect='auto')
ax.axis('off')
ax.text(0.5, 1.02, 'type', size=10, va='bottom', ha='center', transform=ax.transAxes, rotation=90)

for yt, y in yprime_seq_order.reset_index(drop=True).groupby('type').apply(lambda x: np.mean(x.index)).items():
    ax.text(0, y, yt, ha='center', va='center', size=6)

# Estimates source
ax = fig.add_subplot(gs[2])
dat = collection_yprime.loc[S_seq_order, 'raw_reads'].astype(int).values.reshape(-1, 1)
ax.imshow(dat, cmap='Blues', aspect='auto')
ax.axis('off')
ax.text(0.5, 1.02, 'raw\nreads', size=10, va='bottom', ha='center', transform=ax.transAxes, rotation=90)

# Clade
ax = fig.add_subplot(gs[3])
dat = collection_yprime.loc[S_seq_order, 'clade'].replace(dict(zip(v.clades_unique['Clades'], v.clades_unique['clade_no']-1)))\
.astype(int).values.reshape(-1, 1)
ax.imshow(dat, cmap=clades_cmap, aspect='auto')
ax.axis('off')
ax.text(0.5, 1.02, 'clade', size=10, va='bottom', ha='center', transform=ax.transAxes, rotation=90)

# Wine subclade

ax = fig.add_subplot(gs[4])
dat = np.array([s in S_wine_subclade for s in S_seq_order]).reshape(-1, 1)
ax.imshow(dat, cmap='Reds', aspect='auto')
ax.axis('off')
ax.text(0.5, 1.02, 'wine\nsubclade', size=10, va='bottom', ha='center', transform=ax.transAxes, rotation=90)

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}FigS3A.{ext}', dpi=300)
    
plt.show()
plt.close()

## Fig S3B

In [ ]:
fig = plt.figure(figsize=[6,12])
gs = plt.GridSpec(ncols=5, nrows=1, width_ratios=[15,1,1,1,4], wspace=0.1, 
                  left=0.12, right=0.96, top=0.93, bottom=0.05)


ax = fig.add_subplot(gs[0])
x, y, w, h = ax.get_position().bounds
ax_cbar = fig.add_axes([x*1.05, (y+h)*1.03, w*0.5, h*0.01])

dat = yprime_seq_order.value_counts(['type', 'strain_assembly']).reset_index().pivot_table(index='strain_assembly', columns='type', values='count').fillna(0)

S = collection_yprime.sort_values(by=['clade_no', 'strain']).index
S_raw_reads = collection_yprime.loc[collection_yprime['raw_reads']].index

for s in S:
    if s not in dat.index:
        dat.loc[s] = 0
    if s in S_raw_reads:
        dat.loc[s] = dat.loc[s] * 0.2

dat = dat.loc[S, yprime_types_int]
dat[dat==0] = np.nan
dat = np.log10(dat).fillna(0)
vmin = 0
vmax = dat.values.flatten().max()

sns.heatmap(dat, cmap='viridis', ax=ax, vmin=vmin, vmax=vmax, 
            cbar_ax=ax_cbar, cbar_kws=dict(orientation='horizontal'))

#ax.set_
ax.set_yticks(np.arange(len(S))+0.5, labels=S)
ax.tick_params(axis='y', which='major', labelsize=4)


cb_x_ticks = np.linspace(vmin, vmax, 5)
ax_cbar.set_xticks(cb_x_ticks)
ax_cbar.set_xticklabels([f'{10**x:.0f}' if x>0 else r'$\leq1$' for x in cb_x_ticks])
ax_cbar.text(1.05, 0.5, 'Y\' CN', 
             ha='left', va='center', transform=ax_cbar.transAxes)

# Estimates source
ax = fig.add_subplot(gs[1])
dat = collection_yprime.loc[S, 'raw_reads'].astype(int).values.reshape(-1, 1)
ax.imshow(dat, cmap='Blues', aspect='auto')
ax.axis('off')
ax.text(0.5, 1.02, 'raw\nreads', size=10, va='bottom', ha='center', transform=ax.transAxes, rotation=90)
print(dat.shape)
# Clade
ax = fig.add_subplot(gs[2])
dat = collection_yprime.loc[S, 'clade'].replace(dict(zip(v.clades_unique['Clades'], v.clades_unique['clade_no']-1)))\
.astype(int).values.reshape(-1, 1)
ax.imshow(dat, cmap=clades_cmap, aspect='auto')
ax.axis('off')
ax.text(0.5, 1.02, 'clade', size=10, va='bottom', ha='center', transform=ax.transAxes, rotation=90)

ax = fig.add_subplot(gs[4])
ax.barh(np.arange(len(S)), collection_yprime.loc[S, 'y_prime_cn_estimate'], height=1, color='k')
ax.set_yticks(np.arange(len(S)), labels=S)
ax.set_ylim(-0.5, len(S)-0.5)
ax.tick_params(axis='y', which='major', labelsize=4)
ax.invert_yaxis()
ax.set_xlabel('Y\' CN')
sns.despine(ax=ax)

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}FigS3B.{ext}', dpi=300)
    
plt.show()
plt.close()

In [ ]:
# export sequence IDs for the yprime types
for yt, df in yprime_seq_order.groupby('type'):
    # exclude the ones that are too short
    if yt not in [14, 15, 16, 17]:
        #df['name'].to_csv(f'{base_dir}yprime_consensus/yprime.{yt}.IDs.txt', index=False, header=False)
        ...

# Import the consensus sequences alignment

In [ ]:
aln_cons_seq = {}
aln_cons_bool = {}

for seq in SeqIO.parse(f'{v.base_dir}yprime_consensus/combined.yprime.trim_cons.aln.fasta', 'fasta'):
    aln_cons_seq[seq.id] = np.array(list(seq.seq))
    aln_cons_bool[seq.id] = aln_cons_seq[seq.id] != '-'

aln_cons_seq = pd.DataFrame(aln_cons_seq).T
aln_cons_bool = pd.DataFrame(aln_cons_bool).T

## Fig S2D-F

In [ ]:
fig = plt.figure(figsize=[10, 9])
gs = plt.GridSpec(ncols=16, nrows=2, height_ratios=[2,1], 
                  wspace=1, hspace=0.25, left=0.03, top=0.94, bottom=0.06, right=0.97)

## Phylo ##

ax = fig.add_subplot(gs[0,:9])

tree_raxml_path = '/home/mathieu/postdoc_heasley/short_read_project/raxml/GTGTR4_outgroup/\
orf_coding.random500.concat.fasta.raxml.support.icytree_reordered_unquoted'

# get the raw tree to add bootstrap values to the reordered one
tree_raxml_raw_path = '/home/mathieu/postdoc_heasley/short_read_project/raxml/GTGTR4_outgroup/orf_coding.random500.concat.fasta.raxml.support'
tree_raxml_raw =  Phylo.read(tree_raxml_raw_path, 'newick')
tree_bio = Phylo.read(tree_raxml_path, format='newick')

# Prune the tree
for clade in tree_bio.get_terminals():
    if clade.name not in collection_yprime['strain'].values:
        tree_bio.prune(clade)
S_tree = [clade.name for clade in tree_bio.get_terminals()]
SA_tree = list(collection_yprime.set_index('strain').loc[S_tree, 'strain_assembly'].values)
Y = np.arange(len(S_tree))

tree_viz = TreeViz(tree_bio, leaf_label_size=1, orientation='right', 
                   leaf_label_xmargin_ratio=0.03)
tree_viz.set_node_line_props('N_1', lw=1, color='k', descendent=True)

for s in tree_viz.leaf_labels:
    clade = strain_clade_dict[s]
    source = strain_source_dict[s]
    tree_viz.marker(s, marker='o', linewidths=0, alpha=1, size=6, color=v.clade_color[clade])
    tree_viz.set_node_label_props(s, size=6, color=v.source_simplified_color[source])

for s, (x,y) in tree_viz.name2xy_right.items():
    if s in tree_viz.leaf_labels:
        clade = strain_clade_dict[s]
        mfc = v.clades_unique.loc[clade, 'mfc']
        fa = FancyArrow(x, y, 0.05-x, 0, fc=mfc, width=1, head_length=0, head_width=0, lw=0, alpha=0.2, 
                        zorder=-1, clip_on=False)
        ax.add_patch(fa)
    
tree_viz.show_scale_bar(loc='upper center', text_size=10)

tree_viz.plotfig(ax=ax)
ax.set_xlim(0, 0.035)
ax.set_ylim(Y[[0, -1]]+np.array([0.5, 1.5]))


clades_subtree = collection_yprime.loc[SA_tree].sort_values(by='clade_no')['clade'].unique()
legend_clades = [Line2D([0], [0], lw=0, marker='o', ms=6, mfc=v.clade_color[c], mew=0, label=c) for c in clades_subtree]
legend_clades = ax.legend(handles=legend_clades, loc=2, bbox_to_anchor=[-0.03, 1.03], 
                          frameon=False, title='clade')
ax.add_artist(legend_clades)

## Seq before

ax = fig.add_subplot(gs[0, 9])

dat = strains_info_sr.loc[S_tree, ['seq_before','seq_spore']].replace({'seq_before':{False:0.9, True:0.1}, 'seq_spore':{False:0, True:0.4}})\
    .sum(axis=1).values.reshape(-1,1).astype(float)
mask_ref = strains_info_sr.loc[S_tree, 'seq_before'].isna()
dat = np.ma.masked_where(mask_ref.values.reshape(-1,1), dat)
hm_seqbefore = ax.imshow(dat, vmin=0, vmax=1, cmap='binary', aspect='auto', interpolation='nearest')

for y in np.where(mask_ref)[0]:
    ax.scatter([0], [y], s=24, c='k', 
               linewidths=0.5, marker=(6,2,0), clip_on=False)
#hm_seqbefore = ax.imshow(dat, vmin=0, vmax=1, cmap='binary', aspect='auto', interpolation='nearest')

ax.set_ylim(Y[[0, -1]]+np.array([-0.5, 0.5]))
ax.invert_yaxis()
ax.axis('off')
ax.set_title('prev.\nseq', size=10)

## Ploidy

ax = fig.add_subplot(gs[0, 10])

dat = strains_info_sr.loc[S_tree, ['ploidy']].values.reshape(-1,1).astype(float)

hm_ploidy = ax.imshow(dat, vmin=0, vmax=4, cmap='Greens', aspect='auto', interpolation='nearest')

ax.set_ylim(Y[[0, -1]]+np.array([-0.5, 0.5]))
ax.invert_yaxis()
ax.axis('off')
ax.set_title('ploidy', size=10)

## CN barplot

ax = fig.add_subplot(gs[0, 12:])

dat = collection_yprime.groupby('strain')['y_prime_len'].mean().loc[S_tree].values*1e-3

ax.barh(Y, dat, height=1, color=[colormaps.get_cmap('viridis')(i/1.4e3) for i in dat])
ax.set_ylim(Y[[0, -1]]+np.array([-0.5, 0.5]))
ax.set_yticks(np.arange(len(S_tree)), labels=S_tree, size=6)

for yt in ax.get_yticklabels():
    s = yt.get_text()
    source = strain_source_dict[s]
    c = v.source_simplified_color[source]
    yt.set_color(c)    
    
ax.invert_yaxis()
ax.set_xlabel('Y\' kb')
sns.despine(ax=ax)

## Dendrogram ##

ax = fig.add_subplot(gs[1,:2])
x, y, w, h = ax.get_position().bounds
ax.axis('off')
ax = fig.add_axes((x, y, w*1.25, h))

pdist_jci = pdist(aln_cons_seq.values, metric=lambda u,v: (u!=v).sum())
lnk_jci = hierarchy.linkage(pdist_jci, method='complete')
dendro = hierarchy.dendrogram(lnk_jci, truncate_mode=None, orientation='left', ax=ax)
clustered_consensus_order = aln_cons_seq.index[dendro['leaves']]

for coll in ax.collections:
    coll.set_color('k')
    coll.set_linewidth(1)
ax.set_yticklabels([])
ax.invert_yaxis()
ax.set_xticks([])
for sp in ax.spines.values():
    sp.set_visible(False)


## MSA ##

ax = fig.add_subplot(gs[1,2:7])

dat = aln_cons_seq.map(lambda x: v.atcg_num[x]).loc[clustered_consensus_order]
ax.imshow(dat, cmap=v.atcg_cmap, aspect='auto', interpolation='nearest', zorder=1)

# add sequence subdivisions
for i in range(17):
    ax.axhline(i+0.5, c='w', lw=1.5, zorder=2)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_yticks([])
ax.set_xlim(0, 7500)
ax.set_xticks(np.arange(0, 8000, 1000))
ax.set_xticklabels(np.arange(0, 8, 1))
ax.set_xlabel('position (kb)')

for y,l in enumerate(clustered_consensus_order):
    t = ax.text(7200, y, v.at_alias.get(l, l), size=7, va='center', ha='left')
    
    c = v.at_color.get(l, 'k')
    r, g, b = colors.to_rgb(c)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    
    t.set_backgroundcolor(c)
    t.get_bbox_patch().set_boxstyle('square', pad=0.15)
    if l < 0.5:
        t.set_color('w')

## CN heatmap ##

ax = fig.add_subplot(gs[1,8:])

dat = yprime_seq_order.value_counts(['yprime_type', 'strain_assembly']).reset_index()\
    .pivot_table(index='yprime_type', columns='strain_assembly', values='count').fillna(0)
dat.loc['pYP1-L1'] = 0

S = collection_yprime.sort_values(by=['clade_no', 'strain']).index
S_raw_reads = collection_yprime.loc[collection_yprime['raw_reads']].index

for s in S:
    if s not in dat.columns:
        dat[s] = 0
    if s in S_raw_reads:
        dat[s] *= 0.2

dat = dat.loc[clustered_consensus_order, SA_tree]
dat[dat==0] = np.nan
dat = np.log10(dat).fillna(0)
vmin = 0
vmax = dat.values.flatten().max()

hm_cn = ax.imshow(dat, vmin=vmin, vmax=vmax,
                  cmap='viridis', aspect='auto')
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_yticks([])
ax.set_ylim(ax.get_ylim())
ax.set_xticks(np.arange(len(SA_tree)))
ax.set_xticklabels(collection_yprime.loc[SA_tree, 'strain'], size=5, rotation=90)
ax.set_xlabel('')
X = np.arange(len(SA_tree))
ax.scatter(X, np.repeat(18, len(SA_tree)), s=9, zorder=2,
           c=collection_yprime.loc[SA_tree, 'clade'].apply(lambda c: v.clade_color[c]),
          clip_on=False)

x, y, w, h = ax.get_position().bounds
ax_cbar = fig.add_axes([x*1.05, (y+h)*1.1, w*0.25, h*0.05])
cb = plt.colorbar(hm_cn, cax=ax_cbar, orientation='horizontal')
cb.outline.set_visible(False)
cb_x_ticks = np.linspace(vmin, vmax, 4)
ax_cbar.set_xticks(cb_x_ticks)
ax_cbar.set_xticklabels([f'{10**x:.0f}' if x>0 else r'$\leq1$' for x in cb_x_ticks])
ax_cbar.text(1.05, 0.5, 'Y\' CN', 
             ha='left', va='center', transform=ax_cbar.transAxes)


fig.text(0.01, 0.97, 'D', fontweight='bold', size=24)
fig.text(0.01, 0.36, 'E', fontweight='bold', size=24)
fig.text(0.48, 0.36, 'F', fontweight='bold', size=24)

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}FigS2D-F.{ext}', dpi=300)

#plt.show()
plt.close()

# Agreement between clustering and alignment

In [ ]:
YPRIME_TYPES_COMPARISON = []

for s, df in yprime_seq_order.loc[yprime_seq_order['yprime_type'].isin(v.yprime_types)].groupby('strain_assembly'):
    if collection_yprime.loc[s, 'raw_reads'] == True:
        
        ref = df.rename({'query':'seqnames', 'start':'starts','end':'ends'}, axis=1)
        mcols_ref = BiocFrame.from_pandas(ref[['name','yprime_type']])
        gr_ref = GenomicRanges.from_pandas(ref[['seqnames', 'starts', 'ends', 'strand']])
        gr_ref = gr_ref.set_mcols(mcols_ref)

        for (tg_alias, tg) in zip(['regular', 'TG'], ['', '_TG']):
        
            paf = v.parse_paf(f'{v.base_dir}data/paf/{s}_50X.aln.S288C_masked.repeats.subtypes{tg}.paf')
            paf['yprime'] = paf['subject'].isin(v.yprime_types)
            paf['TG_repeat'] = paf['subject'] == 'TG_repeat'
            TG_sum = ((paf['TG_repeat']) & (paf[16]=='tp:A:P')).sum()
            paf = paf.loc[(paf[16]=='tp:A:P') & (paf['yprime'])]
            paf = paf.reset_index().rename({'rid':'seqnames', 'qstart':'starts','qend':'ends','subject':'yprime_type'}, axis=1)
            #paf['type'] = paf['subject'].apply(lambda x: int(re.match(yprime_type_pattern, x).group(1)))
            paf['strand'] = paf['strand'].replace({1:'+', -1:'-'})
            
            mcols_paf = BiocFrame.from_pandas(paf[['yprime_type']])
            gr_paf = GenomicRanges.from_pandas(paf[['seqnames', 'starts', 'ends', 'strand']])
            gr_paf = gr_paf.set_mcols(mcols_paf)
        
            gr_overlaps = gr_ref.find_overlaps(gr_paf)
            names_gff = gr_ref[gr_overlaps['self_hits']].get_mcols()['name']
            types_gff = gr_ref[gr_overlaps['self_hits']].get_mcols()['yprime_type']
            types_paf = gr_paf[gr_overlaps['query_hits']].get_mcols()['yprime_type']
        
            df = pd.DataFrame([names_gff, types_gff, types_paf]).T
            df.columns = ['Name', 'type_gff', 'type_paf']
            df['strain'] = s
            df['mapping'] = tg_alias
            df['TG_sum'] = TG_sum
    
            YPRIME_TYPES_COMPARISON.append(df)

YPRIME_TYPES_COMPARISON = pd.concat(YPRIME_TYPES_COMPARISON).reset_index(drop=True)

In [ ]:
S = v.strains_info.sort_values(by='Y\' element').index
S = [s for s in S if s in YPRIME_TYPES_COMPARISON['strain'].values]

fig = plt.figure(figsize=[15,8])
gs = plt.GridSpec(ncols=5, nrows=1, width_ratios=[1,2,2,5,5], wspace=0.5)

ax = fig.add_subplot(gs[0])
sns.heatmap(v.strains_info.loc[S, ['Y\' element']], cmap='viridis',
            annot=True, annot_kws={'size':6}, fmt='.0f', cbar=False, ax=ax)
ax.set_title('Y\' CN')

ax = fig.add_subplot(gs[1])
dat = YPRIME_TYPES_COMPARISON.groupby(['strain','mapping'])['TG_sum'].mean()\
    .rename('TG_sum').reset_index().pivot_table(index='strain', columns='mapping', values='TG_sum')

sns.heatmap(dat.loc[S, ['regular','TG']], cmap=sns.light_palette('dodgerblue', as_cmap=True), vmin=0, vmax=10000,
            annot=True, annot_kws={'size':6}, cbar=False, fmt='.0f', ax=ax)
ax.set_title('# TG annots')

# classification result
ax = fig.add_subplot(gs[2])
dat = YPRIME_TYPES_COMPARISON.groupby(['strain','mapping']).apply(lambda x: (x['type_gff']==x['type_paf']).mean(), include_groups=False)\
    .rename('matching_subtypes').reset_index().pivot_table(index='strain', columns='mapping', values='matching_subtypes')

sns.heatmap(dat.loc[S, ['regular','TG']], cmap='Greens', vmin=0, vmax=1,
            annot=True, annot_kws={'size':6}, cbar=False, fmt='.2f', ax=ax)
ax.set_title('Y\' subtypes\nclassification')

ax = fig.add_subplot(gs[3])

dat = YPRIME_TYPES_COMPARISON.set_index('mapping').loc['regular'].groupby(['strain','mapping'])['type_gff'].value_counts()\
    .reset_index().pivot_table(index='strain', columns='type_gff', values='count').fillna(0)
#dat = dat.replace({0:np.nan})
sns.heatmap(dat.loc[S, v.yprime_types], cmap='viridis', annot=True, fmt='.0f', 
            annot_kws={'size':6}, vmin=0, vmax=500, cbar=None, ax=ax)
ax.set_title('# Y\' subtypes GFF')

ax = fig.add_subplot(gs[4])

dat = YPRIME_TYPES_COMPARISON.set_index('mapping').loc['regular'].groupby(['strain','mapping'])['type_paf'].value_counts()\
    .reset_index().pivot_table(index='strain', columns='type_paf', values='count').fillna(0)
for yt in v.yprime_types:
    if yt not in dat.columns:
        dat[yt] = 0
#dat = dat.replace({0:np.nan})
sns.heatmap(dat.loc[S, v.yprime_types], cmap='viridis', annot=True, fmt='.0f', 
            annot_kws={'size':6}, vmin=0, vmax=500, cbar=None, ax=ax)
ax.set_title('# Y\' subtypes PAF')

for ax in fig.axes:
    ax.set_yticks(np.arange(len(S))+0.5)
    ax.set_yticklabels(S, size=7)
    ax.set_xlabel('')

plt.show()
plt.close()

In [ ]:
df = pd.DataFrame(np.zeros((len(yprime_types), len(yprime_types))), index=yprime_types, columns=yprime_types)
for i in YPRIME_TYPES_COMPARISON.loc[YPRIME_TYPES_COMPARISON['mapping']=='regular'].index:
    y1, y2 = YPRIME_TYPES_COMPARISON.loc[i, ['type_gff', 'type_paf']]
    df.loc[y1, y2] += 1

fig, ax = plt.subplots()
sns.heatmap(df, cmap='Blues', ax=ax)
ax.set_ylabel('GFF')
ax.set_xlabel('PAF')

plt.show()
plt.close()

In [ ]:
s = 'YJM311'
paf = parse_paf(f'{base_dir}data/paf/{s}_50X.aln.S288C_masked.repeats.subtypes.paf').set_index('rid')
paf['strand'] = np.where(paf['strand']=='+', 1, -1)
paf['yprime'] = paf['subject'].isin(yprime_types)
paf['subtelo'] = paf['subject'].isin(at_subtelomeres)
paf['nuc'] = paf['subject'].isin(nuclear_chromosomes)
paf = paf.loc[paf[16]=='tp:A:P']

for rid in YPRIME_TYPES_COMPARISON.set_index(['mapping', 'strain']).loc[('regular', s)].value_counts('rid').iloc[:10].index:
#rid = '99479902-0808-4ddc-8faf-6655535b55bc'

    sub_paf = paf.loc[rid].sort_values(by='qstart').reset_index()
    
    fig, ax = plt.subplots(figsize=[12,5])
    
    for y in (1,2):
        ax.plot([0, sub_paf.loc[0, 'qlen']], [y,y], c='k')
    
    # plot PAF hits
    y = 1
    
    for i in sub_paf.index:
        qstart, qend, strand, subject, sstart, send, nuc, subtelo, yprime = sub_paf.loc[i, ['qstart','qend','strand','subject','sstart','send','nuc','subtelo','yprime']]
        X = [qstart, qend][::strand]
        arrow_length = np.abs(qend-qstart)
    
        if subject == 'TG_repeat':
            fa = FancyArrow(X[0], y, (X[1]-X[0]), 0, width=0.8, fc=at_color[subject], lw=0, zorder=2,
                            length_includes_head=False, head_width=0.8, head_length=0)
        
        elif subtelo:
            fa = FancyArrow(X[0], y, (X[1]-X[0]), 0, width=0.8, fc=at_color[subject], lw=0, zorder=2,
                            length_includes_head=True, head_width=0.8, head_length=np.min([0.3*repeats_lengths[subject], arrow_length]))
            if yprime:
                subtype = subject.split('.')[1]
                ax.text(np.mean(X), y, subtype, ha='center', va='center')
            
        elif nuc:
            fa = FancyArrow(X[0], y, (X[1]-X[0]), 0, width=0.6, fc='grey', lw=0, zorder=2, alpha=0.6,
                            length_includes_head=True, head_width=0.8, head_length=np.min([1000, arrow_length]))
            
        else:
            fa = FancyArrow(X[0], y, (X[1]-X[0]), 0, width=0.6, fc='gold', lw=0, zorder=2,
                            length_includes_head=True, head_width=0.8, head_length=np.min([100, arrow_length]))
        
        ax.add_patch(fa)        
    
    
    y = 2
    dat = YPRIME_TYPES_COMPARISON.set_index(['mapping', 'rid']).loc[('regular', rid)].reset_index(drop=True)
    
    for i in dat.index:
        qstart, qend, strand, type_gff = dat.loc[i, ['start', 'end', 'strand', 'type_gff']]
        if strand == '+':
            strand = 1
        else:
            strand = -1
        subject = f'yprime.{type_gff}.trim_cons'
        X = [qstart, qend][::strand]
        fa = FancyArrow(X[0], y, (X[1]-X[0]), 0, width=0.8, fc=at_color[subject], lw=0, zorder=2,
                            length_includes_head=True, head_width=0.8, head_length=np.min([0.3*repeats_lengths[subject], arrow_length]))
        ax.add_patch(fa)   
        ax.text(np.mean(X), y, type_gff, ha='center', va='center')
    
    ax.set_ylim(0,3)
    ax.set_title(rid)

    ax.text(0, 1, 'PAF', size=14, ha='right', va='center')
    ax.text(0, 2, 'GFF', size=14, ha='right', va='center')
    
    plt.show()
    plt.close()